# ERA5 pressure levels: quickstart

A minimal pull of ERA5 pressure-level data: one hour of 500 hPa temperature
over the UK, opened with xarray and plotted as a map. See
[`docs/era5-pressure-levels/README.md`](../docs/era5-pressure-levels/README.md)
for the full reference.

Prerequisites are the same as ERA5 single levels: a CDS account, accepted
licence for this dataset, and `~/.cdsapirc` credentials. If you have already
set up the single-levels notebook, you need only accept the pressure-levels
licence separately before the API will serve this dataset.

In [ ]:
# ==================================================================
# USER CONFIGURATION - Edit these values for your use case
# ==================================================================
VARIABLES = ["temperature"]
PRESSURE_LEVELS = ["500"]  # hPa, as strings
YEARS = ["2023"]
MONTHS = ["07"]
DAYS = ["01"]
HOURS = ["12:00"]
BBOX = [55, -8, 49, 2]  # [north, west, south, east] - UK
OUTPUT_DIR = "../data/era5-pressure-levels"
OUTPUT_FILENAME = "era5_pressure_levels_quickstart.nc"
# ==================================================================

## Imports and environment check

Same environment bootstrap as the single-levels notebook: version-print and
credential check.

In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

import cdsapi
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

print(f"Python       {sys.version.split()[0]}")
for pkg in ["cdsapi", "xarray", "matplotlib", "cartopy", "netcdf4"]:
    print(f"{pkg:12} {version(pkg)}")

def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / "CLAUDE.md").exists():
            return parent
    raise RuntimeError("Could not find repo root.")

repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from common.credentials import check_cds_credentials
check_cds_credentials()
print("\nCDS credentials found.")

## Submit the request

The key difference from single levels: the request dictionary includes a
`pressure_level` field. All 37 ERA5 pressure levels are available for every
variable.

In [ ]:
output_path = Path(OUTPUT_DIR) / OUTPUT_FILENAME
output_path.parent.mkdir(parents=True, exist_ok=True)

request = {
    "product_type": ["reanalysis"],
    "variable": VARIABLES,
    "pressure_level": PRESSURE_LEVELS,
    "year": YEARS,
    "month": MONTHS,
    "day": DAYS,
    "time": HOURS,
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": BBOX,
}

client = cdsapi.Client()
client.retrieve("reanalysis-era5-pressure-levels", request).download(str(output_path))
print(f"Saved: {output_path} ({output_path.stat().st_size / 1e6:.2f} MB)")

## Open and inspect

Pressure-level NetCDF output has a `pressure_level` coordinate alongside the
usual `valid_time`, `latitude`, `longitude`. The temperature variable
appears as `t` (same as the GRIB short name, since `t` does not start with
a digit).

In [ ]:
ds = xr.open_dataset(output_path)
print(ds)

## Plot a map

Plot 500 hPa temperature over the UK, converted to Celsius. 500 hPa is
roughly 5.5 km altitude, well above the boundary layer, and sits near the
mean atmospheric level for weather systems.

In [ ]:
t = ds["t"].squeeze() - 273.15

fig = plt.figure(figsize=(10, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

t.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="RdBu_r",
    cbar_kwargs={"label": "500 hPa temperature (C)"},
)

ax.coastlines(resolution="50m", linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor="gray")
gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
gl.top_labels = False
gl.right_labels = False

ax.set_title(f"ERA5 500 hPa temperature, {YEARS[0]}-{MONTHS[0]}-{DAYS[0]} {HOURS[0]} UTC")
plt.tight_layout()
plt.show()

## Next steps

- **Full reference**: [`docs/era5-pressure-levels/README.md`](../docs/era5-pressure-levels/README.md)
- **Multiple levels**: set `PRESSURE_LEVELS = ["200", "500", "850"]` and plot a vertical section with `ds["t"].plot(y="pressure_level", yincrease=False)`.
- **Surface-level variables**: see the [ERA5 single levels](../notebooks/era5_single_levels_quickstart.ipynb) entry.
- **Geopotential height**: request `geopotential` and divide by 9.80665 to convert from m2 s-2 to metres.